# Presentation of our solution -- Quantum urban vision

Today's world is obeserved through the lens of multiple devices. Here, we consider images of our earth took from satellites. All sort of things can be spotted through these images. The idea is to train a quantum model to flag some target components that we could find on these images. Here, for simplicity, we will consider a narrower dataset. 

We assume that our model can only input 10 type of images, all with the same dimensions: planes, airports, ships, harbor, tennis court, basketball court, clouds, dense residential, railway station, river. Our goal is to follow the traffic both in the air and in the water. That means that our model should flag us when it sees any airport, ship, plane, or harbor. 

Now, knowing an image has either a ship or a plane is already good, but not sufficient, since these two are broadly different. We can thus train another model that takes in entry an image of either a plane, a boat, a harbor, or an airport, and that classifies the image in its corresponding class. 

To train our models, we used an already existing dataset called `NWPU-RESISC45`. It contains images from satellites classified in 45 classes. We cutted all classes but the previously mentionned classes of interest, and use them as training data and testing data. 

### A sneak peak of our classification model


First, let's see how our model does the first classification, outputing a binary answer: either flagged or not flagged image.

In [ ]:
from trainer import get_datas, rafine_y_values, QuantumReservoirNet, train, acc

import torch.nn as nn
import torch
import torch.optim as optim

# nb_classes = 4
anomaly_ratio = 1

x_train, y_train, x_test, y_test = get_datas(400, anomaly_ratio)

# Refine y values to be between 0 and nb_classes-1
y_train, y_test = rafine_y_values(y_train, y_test)

# Convert to PyTorch tensors
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Define the model, loss function and optimizer
model = QuantumReservoirNet()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
batch_size = 16

model = train(
    model,
    x_train_t,
    y_train_t,
    x_test_t,
    y_test_t,
    criterion,
    optimizer,
    epochs=epochs,
    batch_size=batch_size,
)


NameError: name 'get_datas' is not defined

Second, we can see how our model does when it comes to classify between 4 images. 

In [1]:
from anomaly_detector import create_and_train_classifier, create_anomaly_dataset, extract_image_features, feature_reductor, create_quantum_kernels, detect_anomaly
from sklearn.metrics import roc_auc_score
import numpy as np


# ====================================================
# DATA
# ====================================================

train_tensor = create_anomaly_dataset(0.0, 100)

x_train = train_tensor[0][0]
y_train = train_tensor[0][1]

test_tensor = create_anomaly_dataset(0.1, 100)

x_test = test_tensor[1][0]
y_test = test_tensor[1][1]


# ====================================================
# FEATURE EXTRACTION
# ====================================================

train_features = extract_image_features(x_train)
test_features = extract_image_features(x_train)

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)


# ====================================================
# PCA + scaler -> 8 DIMENSIONS
# ====================================================

train_pca = feature_reductor(train_features, num_features=8)
test_pca = feature_reductor(train_features, num_features=8)


# ====================================================
# QUANTUM KERNEL
# ====================================================

train_kernel, test_kernel = create_quantum_kernels(x_train=train_pca, x_test=test_pca, num_features=8)


# ====================================================
# ONE-CLASS SVM
# ====================================================
# Unlike standard SVMs that require data from two classes to find a separating boundary, a One-Class SVM trains only on "normal" data to learn its distribution, flagging anything that deviates significantly from this norm
ocsvm = create_and_train_classifier(train_kernel)


# ====================================================
# ANOMALY SCORES
# ====================================================

decision_scores, preds = detect_anomaly(ocsvm, test_kernel)


# ====================================================
# CONFUSION MATRIX COMPONENTS
# ====================================================

y_test=y_test.cpu().numpy()  # Convert to NumPy array if it's a PyTorch tensor

TP = np.sum((preds == 1) & (y_test == 1))
FP = np.sum((preds == 1) & (y_test == 0))
TN = np.sum((preds == 0) & (y_test == 0))
FN = np.sum((preds == 0) & (y_test == 1))

print("\n==============================")
print("CONFUSION MATRIX SUMMARY")
print("==============================")
print(f"True Positives  (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives  (TN): {TN}")
print(f"False Negatives (FN): {FN}")


Train features: (100, 512)
Test features: (100, 512)
Building training kernel matrix...
Building test kernel matrix...

ROC-AUC: 0.5833
Sample 000 | Score=0.000062 | Label=0
Sample 001 | Score=-0.000306 | Label=0
Sample 002 | Score=-0.000025 | Label=0
Sample 003 | Score=-0.000018 | Label=0
Sample 004 | Score=0.000500 | Label=0
Sample 005 | Score=-0.000104 | Label=0
Sample 006 | Score=-0.000115 | Label=0
Sample 007 | Score=0.000177 | Label=0
Sample 008 | Score=0.000071 | Label=0
Sample 009 | Score=-0.000090 | Label=0
Sample 010 | Score=0.000440 | Label=0
Sample 011 | Score=-0.000070 | Label=0
Sample 012 | Score=0.000209 | Label=0
Sample 013 | Score=0.000058 | Label=0
Sample 014 | Score=-0.000092 | Label=1
Sample 015 | Score=-0.000368 | Label=0
Sample 016 | Score=0.000038 | Label=0
Sample 017 | Score=0.000080 | Label=0
Sample 018 | Score=-0.000134 | Label=0
Sample 019 | Score=-0.000389 | Label=1
Sample 020 | Score=-0.000447 | Label=0
Sample 021 | Score=-0.000212 | Label=0
Sample 022 | Sc

### The full pipeline and the role of quantum in it
The full pipeline can be understood by thinking of a satellite image of size 32x32 coming as an input to the algorithm. 

The first step is to process the image into 512 characteristic features of the image, using Resnet 18, a pretrained model available. We then use a PCA 10 algorithm to reduce to the 10 most important features. Then, we use these in a QSVM, algorithm, marking the first use of a quantum computer. This QSVM is only trained to say wheter or not an image is flagged as normal, we call it a QSVM 1 class. Thus, we have no idea of which class we obtained. We then need to use another algorithm, which is only applied if the image is flagged as anormal by the QSVM.

The same entry image is thus converted into 10 new features using a classical neural network. These 10 features are used in a photonic circuit that performs a boson sampling method to classify the data. Then, the algorithm as a return value of either `"not_flagged"`, `"airport"`, `"harbor"`, `"ship"`, or `"airplane"`.

Quantum is used as a machine learning engine. All of these task could be done via classical algorithms...

<img src="./pipeline.png" alt="Alt Text" width="1000">